# twinfer on Colab

Builds the CUDA version, runs the tests, compares against Hugging Face transformers on the real
SmolLM2-135M-Instruct, and prints the benchmark tables for the README.

Switch the runtime to a GPU first (Runtime > Change runtime type > T4 GPU).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2

In [ ]:
import os
if not os.path.exists("twinfer"):
    !git clone -q https://github.com/Yashvardhan0505/twinfer.git
%cd twinfer

## Build and test

In [ ]:
!make -j2 cpu cuda ARCH=native 2>&1 | tail -3
!./build/test-kernels

In [ ]:
!python3 tests/run_tests.py --cpu --cuda

## Real model vs Hugging Face

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("HuggingFaceTB/SmolLM2-135M-Instruct", local_dir="models/smollm2-135m",
                  allow_patterns=["*.json", "*.jinja", "*.safetensors"])
!python3 tools/hf_compare.py models/smollm2-135m --cuda

In [ ]:
!./build/twinfer-cuda -m models/smollm2-135m -p "Explain warp divergence to a beginner in three sentences." -n 200

## Throughput (paste this into the README)

In [ ]:
import re, subprocess
MODEL = "models/smollm2-135m"
runs = [("CUDA fp16 + graph", ["./build/twinfer-cuda"]),
        ("CUDA fp16, no graph", ["./build/twinfer-cuda", "--no-graph"]),
        ("CUDA int8 + graph", ["./build/twinfer-cuda", "--int8"]),
        ("CPU fp32", ["./build/twinfer-cpu"])]

def bench(cmd, *args):
    out = subprocess.run([cmd[0], "-m", MODEL, *args, *cmd[1:]], capture_output=True, text=True, check=True).stdout
    return {k: float(v) for k, v in re.findall(r"(\w+)=([\d.]+)", out)}

print("| backend | prefill tok/s (512-token prompt) | decode tok/s | ms/token |")
print("|---|---|---|---|")
for label, cmd in runs:
    dec = bench(cmd, "--bench", "64" if "cpu" in cmd[0] else "256")  # short prompt: decode speed
    pre = bench(cmd, "--bench", "1", "--bench-prompt", "512")        # long prompt: prefill speed
    print(f"| {label} | {pre['prefill_tok_s']:.0f} | {dec['decode_tok_s']:.0f} | {dec['ms_per_token']:.2f} |")

## Kernels vs cuBLAS, and the roofline

In [ ]:
!./build/bench-kernels --csv bench.csv | tee bench.md

In [ ]:
!mkdir -p docs && python3 tools/roofline.py bench.csv -o docs/roofline.png
from IPython.display import Image
Image("docs/roofline.png")

Download `docs/roofline.png`, commit it, and uncomment the image line in the README.

To profile one decode step:

```
!nsys profile -o decode --force-overwrite true ./build/twinfer-cuda -m models/smollm2-135m --bench 32
!ncu --set full -k regex:gemv_kernel -c 5 -o gemv ./build/twinfer-cuda -m models/smollm2-135m --bench 4 --no-graph
```